# MvDEC S2 with Fig. 2 encoder concatenation inside refinement - REUTERS + 20NEWS

Controlled diagnostic only (`controlled_diagnostic_not_exact_paper_truth`). Same as S2
(`mvdec_2025_scratch_both_views_true_bottleneck_min500_release_l2_v1`) except the fused latent
used by the greedy eigen target, K-Means refresh and L4 is `concat(h_view1, h_view2)` (Fig. 2),
not the Eq. 4 average. The paper states both; this arm tests the Fig. 2 reading end to end
(hypothesis H6 follow-up). A label trajectory sidecar is written per run; it never feeds back.

Kaggle settings: GPU on, Internet on, no input dataset.

In [ ]:
import tensorflow as tf

gpus = tf.config.list_physical_devices("GPU")
print("TensorFlow:", tf.__version__)
print("GPUs:", gpus)
assert gpus, "Enable a Kaggle GPU before running this diagnostic."

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import shutil
import subprocess
import sys

PROTOCOL = "mvdec_2025_scratch_both_views_true_bottleneck_figure2_concat_min500_release_l2_v1"
PINNED_COMMIT = "11a34f57046793b8e12ac136927f7c6fa5cec5c3"
BRANCH = "claude/mvdec-hypotheses-2026-10-24"
REPO_URL = "https://github.com/vdat-18/mvdec_fpmax.git"
REPO = Path("/kaggle/temp/mvdec_fpmax")
OUTPUT = Path("/kaggle/temp/mvdec-s2-concat")
RESULTS = Path("/kaggle/working/mvdec-s2-concat")
RUNS = [("REUTERS", s) for s in range(42, 52)] + [("20NEWS", s) for s in range(42, 47)]

def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


for path in (REPO, OUTPUT, RESULTS):
    if path.exists():
        shutil.rmtree(path)
OUTPUT.mkdir(parents=True)
(RESULTS / "trajectory").mkdir(parents=True)
subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, str(REPO)], check=True)
subprocess.run(["git", "checkout", "-q", PINNED_COMMIT], cwd=REPO, check=True)
TRAINING_COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
assert TRAINING_COMMIT == PINNED_COMMIT, (TRAINING_COMMIT, PINNED_COMMIT)
assert not subprocess.check_output(["git", "status", "--porcelain"], cwd=REPO, text=True).strip()
print("Training commit:", TRAINING_COMMIT)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO), "gdown"], check=True)
sys.path.insert(0, str(REPO / "src"))
subprocess.run(
    [sys.executable, str(REPO / "scripts/fetch_dekm_release.py"), "--datasets", *sorted({d for d, _ in RUNS})],
    cwd=REPO,
    check=True,
)

In [ ]:
subprocess.run(
    [
        sys.executable, "-m", "pytest", "-q",
        "tests/test_mvdec_independent_view2.py",
        "tests/test_mvdec_contract.py",
        "tests/test_data.py",
        "tests/test_mvdec_architecture.py",
        "tests/test_mvdec_trajectory.py",
    ],
    cwd=REPO,
    env={**os.environ, "PYTHONPATH": os.pathsep.join([str(REPO), str(REPO / "src")])},
    check=True,
)

In [ ]:
for dataset, seed in RUNS:
    command = [
        sys.executable, "-u", "src/representation_learning/MVDEC_dense.py", dataset,
        "--runs", "1", "--seed", str(seed),
        "--protocol", PROTOCOL,
        "--progress-interval", "100",
        "--dataset-root", "external_repos/DEKM/datasets",
        "--public-output-dir", str(OUTPUT),
    ]
    print("Running:", " ".join(command), flush=True)
    trajectory = RESULTS / "trajectory" / f"{dataset}_seed_{seed}.jsonl"
    env = {**os.environ, "MVDEC_LABEL_TRAJECTORY_LOG": str(trajectory)}
    subprocess.run(command, cwd=REPO, env=env, check=False, timeout=4 * 60 * 60)

In [ ]:
import statistics

PAPER = {
    "REUTERS": {"view1": (0.7642, 0.5999), "view2": (0.6501, 0.5129), "fused": (0.7813, 0.6141)},
    "20NEWS": {"view1": (0.4120, 0.4376), "view2": (0.4086, 0.4032), "fused": (0.4163, 0.4543)},
}
rows = []
for path in sorted(OUTPUT.rglob("manifest.json")):
    manifest = json.loads(path.read_text(encoding="utf-8"))
    if manifest.get("protocol_id") != PROTOCOL or manifest.get("status") != "complete":
        continue
    rows.append((manifest["dataset"], manifest["seed"], manifest["metrics"]))
    # Keep only small text records; weights and latents stay in /kaggle/temp.
    target = RESULTS / "runs" / path.parent.relative_to(OUTPUT)
    target.mkdir(parents=True, exist_ok=True)
    for small in path.parent.iterdir():
        if small.is_file() and small.suffix in {".json", ".csv", ".txt"}:
            shutil.copy2(small, target / small.name)
rows.sort(key=lambda row: (row[0], row[1]))
print("complete runs:", len(rows), "of", len(RUNS))
for dataset in sorted({row[0] for row in rows}):
    picked = [metrics for d, _, metrics in rows if d == dataset]
    print()
    print(dataset)
    for d, seed, metrics in rows:
        if d == dataset:
            print(seed, "  ".join("%s %.4f/%.4f" % (k, metrics[k + "_acc"], metrics[k + "_nmi"]) for k in PAPER[dataset]))
    print("rep    mean_ACC best_ACC | paper_ACC | mean_NMI best_NMI | paper_NMI")
    for key, (paper_acc, paper_nmi) in PAPER[dataset].items():
        accs = [m[key + "_acc"] for m in picked]
        nmis = [m[key + "_nmi"] for m in picked]
        print("%-6s %8.4f %8.4f | %.4f | %8.4f %8.4f | %.4f" % (key, statistics.mean(accs), max(accs), paper_acc, statistics.mean(nmis), max(nmis), paper_nmi))

zip_path = Path(shutil.make_archive(str(RESULTS) + "-results", "zip", RESULTS.parent, RESULTS.name))
digest = file_sha256(zip_path)
Path(str(zip_path) + ".sha256").write_text(digest + "  " + zip_path.name + chr(10), encoding="ascii")
print("ZIP:", zip_path)
print("SHA-256:", digest)